# Visuals — per-tier deltas and (later) rank-vs-performance

Reads `runs/*/{model}-{arm}/summary.json` only — never raw responses. Base line comes from `runs/base-v1`; FT/sweep runs appear as they land in `runs/ft-v1` and `runs/sweep-*`. Conventions after vlm-alignment's `visuals.ipynb`.

In [ ]:
# CONFIG
MODEL = "8b"
ARMS = ["story", "literal", "two-stage"]
TAGS = ["base-v1", "ft-v1"]   # add sweep tags here as they land

In [ ]:
import json, pathlib
RUNS = pathlib.Path("..") / "runs"
rows = []
for tag in TAGS:
    for arm in ARMS:
        p = RUNS / tag / f"{MODEL}-{arm}" / "summary.json"
        if not p.exists():
            continue
        for tier, s in json.loads(p.read_text()).items():
            rows.append({"tag": tag, "arm": arm, "tier": tier, **s})
print(f"{len(rows)} (tag, arm, tier) cells loaded")
rows[:2]

In [ ]:
# Per-tier correct% and unparseable% bars, one panel per arm.
import matplotlib.pyplot as plt
tiers = ["normal", "hard", "extra_hard", "order5"]
fig, axes = plt.subplots(1, len(ARMS), figsize=(4 * len(ARMS), 3), sharey=True)
for ax, arm in zip(axes, ARMS):
    for tag in TAGS:
        ys = [next((r["unparseable_pct"] for r in rows if r["tag"] == tag and r["arm"] == arm and r["tier"] == t), None) for t in tiers]
        if any(y is not None for y in ys):
            ax.plot(tiers, ys, marker="o", label=tag)
    ax.set_title(f"{MODEL} · {arm} · unparseable%")
    ax.tick_params(axis="x", rotation=30)
axes[0].legend()
plt.tight_layout()

**Future (Phase 5b/6):** rank-vs-performance plot with the base line, per `training/config.py` RANKS; then the mechanistic files noted in `analysis/README.md`.